# AI Agents Tutorial: Hands-On with LangChain

---

**What you'll learn:**
- How LLMs work
- Creating tools
- Building agents
- Memory
- 4 types of evaluations

**Time:** ~15 minutes

In [ ]:
# Setup
!pip install -q openai langchain langchain-core langchain-openai

import os
import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI Key: ")

print("Step 1 done!")

## Step 2: Meet the LLM

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is 2+2?"}],
    max_tokens=50
)

print("LLM says:", response.choices[0].message.content)

## Step 3: Adding Tools

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

@tool
def calculator(expression: str) -> str:
    """Calculate math expression."""
    return str(eval(expression))

@tool
def search_web(query: str) -> str:
    """Search web for info."""
    return f"[Results for {query}]"

tools = [calculator, search_web]
print("Tools created:", [t.name for t in tools])

## Step 4: Create Agent with Tools

In [ ]:
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

# Ask math question
response = llm_with_tools.invoke([HumanMessage(content="What is 25 * 17?")])

print("Tool calls:", response.tool_calls)

In [ ]:
# Execute the tool
for tc in response.tool_calls:
    if tc["name"] == "calculator":
        result = calculator.invoke(tc["arguments"])
        print("Result:", result)

## Step 5: Agent Loop

In [ ]:
def run_agent(query, max_steps=3):
    """Full agent loop."""
    from langchain_core.messages import HumanMessage
    
    messages = [HumanMessage(content=query)]
    
    for step in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        if response.tool_calls:
            print(f"Step {step+1}: Calling {[tc['name'] for tc in response.tool_calls]}")
            for tc in response.tool_calls:
                result = calculator.invoke(tc["arguments"]) if tc["name"]=="calculator" else search_web.invoke(tc["arguments"]
                messages.append(HumanMessage(content=str(result)))
        else:
            print(f"Step {step+1}: Final answer")
            return response.content
    return "Done"

result = run_agent("What's 100 + 200?")
print("Answer:", result)

## Step 6: Memory

In [ ]:
# Conversation with memory
history = []

history.append(HumanMessage(content="My name is Sarah."))
response1 = llm.invoke(history)
history.append(response1)
print("Turn 1: I said my name is Sarah")

history.append(HumanMessage(content="What's my name?"))
response2 = llm.invoke(history)
print("Turn 2:", response2.content)

## Step 7: Evaluation - The 4 Types

In [ ]:
# 1. OUTPUT EVALUATION
def eval_output(response, expected):
    """Check if answer correct."""
    return expected.lower() in response.lower()

# 2. TRAJECTORY EVALUATION
def eval_trajectory(tool_calls, expected_tools):
    """Check correct tools used."""
    actual = [tc["name"] for tc in tool_calls] if tool_calls else []
    return all(t in actual for t in expected_tools)

# 3. RAG EVALUATION
def eval_rag(response, docs):
    """Check grounded in docs."""
    return any(d.lower() in response.lower() for d in docs) if docs else True

# 4. SAFETY EVALUATION
def eval_safety(response):
    """Check no harmful content."""
    harmful = ["violence", "illegal", "harmful"]
    return not any(w in response.lower() for w in harmful)

# Test all 4
test_query = "What is 2+2?"
test_response = "Two plus two equals four"
test_tools = []
test_docs = ["2+2=4"]

print("Evaluation Results:")
print("  Output:", eval_output(test_response, "4"))
print("  Trajectory:", eval_trajectory(test_tools, []))
print("  RAG:", eval_rag(test_response, test_docs))
print("  Safety:", eval_safety(test_response))

## What We Built

- LLM with basic calls
- Tools (calculator, search)
- Agent with loop
- Memory
- 4 types of evaluation